In [16]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle

In [17]:
ruta_OF = os.path.join("res\\raw", "CPRManufacturingOrder.csv")
print(ruta_OF)

res\raw\CPRManufacturingOrder.csv


In [18]:
columns_OF = ["IDManufacturingOrder", "CodCompany",	"CodManufacturingOrder", "IDArticle", "IDStructure", "Description", "TimeUnitDefault",	"IDStructureType",	"Notes", "Quantity", "IDWareHouse", "PlannedEndDate", "PlannedStartDate", "PlannedEndMRPDate", "ManualEndDate", "Priority", "IDMOSituation", "Color", "Group", "InternalCode", "IDRoute", "AcceptedDate", "RealEndDate", "IDProject", "IDProTask", "IDProMaterial", "AutomaticCodification", "ManualPStartDate", "ManualPEndDate", "IDSite", "Historic", "RowTimestamp", "PlanningDirection", "LastModifiedBy", "QuantityBaseFormula", "IDUnitBaseFormula", "IDArticleVersion", "CreationTimestamp", "RowCreatedBy", "IDMaintenanceOrder", "IDOrderMaterialPrevision",	"IDLaunchedMRP", "AutoBatchAssignSalesOrders"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_OF = pd.read_csv(ruta_OF, sep=";", header=None, names=columns_OF)
print(df_OF.shape)

(18106, 43)


In [19]:
df_OF = df_OF[df_OF["CodCompany"] == 100]
print(df_OF.shape)
display(df_OF.sample(1))

(18106, 43)


,IDManufacturingOrder,CodCompany,CodManufacturingOrder,IDArticle,IDStructure,Description,TimeUnitDefault,IDStructureType,Notes,Quantity,IDWareHouse,PlannedEndDate,PlannedStartDate,PlannedEndMRPDate,ManualEndDate,Priority,IDMOSituation,Color,Group,InternalCode,IDRoute,AcceptedDate,RealEndDate,IDProject,IDProTask,IDProMaterial,AutomaticCodification,ManualPStartDate,ManualPEndDate,IDSite,Historic,RowTimestamp,PlanningDirection,LastModifiedBy,QuantityBaseFormula,IDUnitBaseFormula,IDArticleVersion,CreationTimestamp,RowCreatedBy,IDMaintenanceOrder,IDOrderMaterialPrevision,IDLaunchedMRP,AutoBatchAssignSalesOrders
6185,567ca1ee-55d5-46e6-ad56-c63ae3ddc08e,100,OF2507401,f707dc8d-c5d1-47be-b4c6-4ef1cf35066a,97a49c33-cc1b-4f6d-b2e7-0babe3231bbb,Sonderelektrodenschaft 3B6F994,2,d3bd7234-a0af-4c35-b658-dcd8ac943c67,NaN,2.0,8dc40f3a-f875-4dbf-8844-63282db4196f,2025-07-28 16:00:00.000,2025-07-28 13:33:00.000,2025-07-29 00:00:00.000,2025-07-29 00:00:00.000,0,f57597bd-79b3-4a87-9f62-fdd434de7804,-12771464.0,NaN,CV252928/0198/PE0XA2X0143/14,cf345109-6d07-423f-9a2e-bf48b8c569c8,2025-07-17 00:00:00.000,2025-08-19 00:00:00.000,NaN,NaN,NaN,1,2025-07-28 13:33:00.000,2025-07-28 16:00:00.000,5565bc54-9d9c-4dc7-91e7-bb5e206c9bd7,0,2026-06-16 07:51:36.290,1,JMAU,2.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb,NaN,2025-07-17 12:04:00.073,AFER,NaN,NaN,9dc6edfb-def2-4bc3-ad17-d6e9523b57b0,0


In [20]:
df_OF_iniciales = df_OF.shape[0]

# Limpieza de OF.
Hay muchas ordenes de fabricación, para que el modelo tenga sentido debó moldearlo con los tiempos reales totales, es decir, no me sirve trabajar con OF que no estan terminadas, o que tienen imputaciones a medias. 

Esto lo haré de dos formas:

1. Descartando todas las OF las cuales su estado (IDMOSituation) sea "Creada". Ya que OF con este estado nunca han recivido imputaciones de tiempo.

2. Descartando todas las OF las cuales su fecha de finalización (RealEndDate) sea antes del 01-08-2026. 

## 1. Descartar OF por su estado
Las fases que su estado sea creadas debemos eliminarlas, el estado "Creada" puede tener dos ID:    
    - ID 100: 7776bcc3-98be-48f1-913a-30bbb40ef0bb   
    - ID 200: 7776bcc3-98be-48f1-913a-30bbb40ef0bb200   

In [21]:
df_OF_Creada = df_OF[df_OF['IDMOSituation'].isin(["7776bcc3-98be-48f1-913a-30bbb40ef0bb", "7776bcc3-98be-48f1-913a-30bbb40ef0bb200"])]
print(f"Total de OFs con estado Creadas (eliminar): {df_OF_Creada.shape[0]}")

df_OF = df_OF[~df_OF['IDMOSituation'].isin(["7776bcc3-98be-48f1-913a-30bbb40ef0bb", "7776bcc3-98be-48f1-913a-30bbb40ef0bb200"])]
print(f"Total de OFs tras filtrar por situación: {df_OF.shape[0]}")

Total de OFs con estado Creadas (eliminar): 1
Total de OFs tras filtrar por situación: 18105


## 2. Decartar OF por su fecha 
- **Decartar OF por su fecha finalización**      

Aquellas OF con fecha de finalización prevista posterior al 01/08/2026 las descartaremos ya que la totalidad de sus fases no estarán terminadas. 

Esto lo hare con la columna `ManualEndDate`. 

Con el equipo técnico hemos determinado que a partir de agosto, no se puede garantizar la finalización de todas sus fases, por ende vamos a prescindir de todas las OFs que tengan la fecha de finalización en agosto.      

In [22]:
df_OF["CreationTimestamp"] = pd.to_datetime(df_OF["CreationTimestamp"])
print(f"Total OFs {df_OF.shape[0]}")

df_OF_validas = df_OF[df_OF["CreationTimestamp"] >= "2025-06-01"]
print(f"Total OFs válidas {df_OF_validas.shape[0]}")
print(f"Total de OFs fuera de fecha {df_OF.shape[0] - df_OF_validas.shape[0]}")

Total OFs 18105
Total OFs válidas 12802
Total de OFs fuera de fecha 5303


- **Decartar OF por su fecha creación**
Aquellas OF con fecha de creación previa al 01/06/2025 las descartaremos ya que el sistema estaba en fase de implementación, esos datos no son fiables. 


In [23]:
df_OF["ManualEndDate"] = pd.to_datetime(df_OF["ManualEndDate"], errors="coerce")
print(f"Total OFs antes del filtro de fecha de finalización: {df_OF_validas.shape[0]}")

df_OF_validas = df_OF_validas[df_OF_validas["ManualEndDate"] < "2026-08-01"]
df_OF_validas = df_OF_validas[df_OF_validas["ManualEndDate"].notnull()]

print(f"Total OFs válidas (ambos filtros): {df_OF_validas.shape[0]}")


Total OFs antes del filtro de fecha de finalización: 12802
Total OFs válidas (ambos filtros): 11983


In [24]:
df_OF_intervalo_tiempos = df_OF_validas.shape[0] 

## 3. Limpieza de Atributos
Vamos a prescindir de los atributos innecesarios. 

In [25]:
def eliminar_columnas(df):
    """Elimina de df_OF las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original)."""

    cols_sobran = [
        'IDStructure', 'TimeUnitDefault','Notes', 'IDWareHouse', 'Priority', 
        'Color', 'Group', 'InternalCode', 'IDRoute', 'AcceptedDate', 'CodCompany',
        'IDProject', 'IDProTask', 'IDProMaterial', 'AutomaticCodification',
        'Historic', 'RowTimestamp', 'PlanningDirection', 'LastModifiedBy',
        'IDMaintenanceOrder', 'IDOrderMaterialPrevision', 'IDLaunchedMRP', 
        'AutoBatchAssignSalesOrders', 'RowCreatedBy', 'CreationTimestamp', 
        'IDArticleVersion', 'IDSite', 'PlannedEndDate', 'PlannedStartDate',	
        'PlannedEndMRPDate', 'ManualPStartDate', 'ManualPEndDate', 'ManualEndDate'	 
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_OF]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_OF.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [26]:
df_OF = eliminar_columnas(df_OF_validas)

Eliminadas 33 columnas. Quedan 10 columnas, 11983 filas.


In [27]:
display(df_OF.head(1))

,IDManufacturingOrder,CodManufacturingOrder,IDArticle,Description,IDStructureType,Quantity,IDMOSituation,RealEndDate,QuantityBaseFormula,IDUnitBaseFormula
1,0008c8eb-104f-4631-8a7c-50b6d1166cc4,OF2509130,6b526dff-7ba3-4251-99fb-96ff8a71d6eb,Elektrodenschaft F704510502739,d3bd7234-a0af-4c35-b658-dcd8ac943c67,3.0,f57597bd-79b3-4a87-9f62-fdd434de7804,2025-09-26 00:00:00.000,3.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb


Guardamos el `df_OF_validas` en el repositorio res//procesed

In [28]:
df_OF.to_parquet("res//procesed/df_OF.parquet")

Revision del notebook

In [29]:
# guardar variables en local
variables = {
    "OF iniciales": df_OF_iniciales,
    "OF en intervalo temporal": df_OF_intervalo_tiempos
}
with open("res/procesed/variables/4.Tratamiento_Datos_OF.pkl", "wb") as f:
    pickle.dump(variables, f) 

In [30]:
tabla_resultados = pd.DataFrame({
    "OF iniciales": [df_OF_iniciales], # nueva variable
    "OF en intervalo temporal": [df_OF_intervalo_tiempos] # nuva variable
}).T

tabla_resultados = tabla_resultados.rename(columns={0: "Número de OF"})

print("=======================================================")
print("Resultados del notebook:")
print("=======================================================")
display(tabla_resultados)
print("=======================================================")

Resultados del notebook:


,Número de OF
OF iniciales,18106
OF en intervalo temporal,11983
